## 20 — CD e rollback do serving (preparação)

Primeiro ensaio blue-green: duas identidades de release, **mesmos pesos e mesma imagem fixada**, gateway NGINX e recursos temporários próprios. O pacote corrompido deve ser bloqueado antes de receber tráfego; B precisa passar readiness/identidade/paridade; uma falha explícita deve produzir 502 e acionar retorno verificado a A. Sem MLflow/S3/datasets ou retreino. **Gate Docker nativo pendente.** Não é publicação automática nem prova de zero downtime.

Ver [procedimento](../../docs/operations/SERVING_ROLLOUT.md).


## Gate explícito no host

O roteiro completo está no [runbook](../../docs/operations/SERVING_ROLLOUT.md). Requer a release já restaurada e a imagem Docker anterior ainda presente. O ensaio cria recursos próprios; não altera o serving em uso nem depende de tracking ativo.

```bash
docker pull nginx:1.30.6-alpine@sha256:970364228b1e4f932abc22d321932816782cca8a20b85a77552a7cfe9acfb3b5
.venv/bin/python -m tox -e serving-rollout
```

O recibo fica em `data/serving/rollouts/<uuid>/result.json`; somente status success com sequência completa, fontes intactas e cleanup concluído aprova o ensaio. Em falha, preservar o recibo. Não repetir treino/holdout/exportação.


### Correção da rede do ensaio de rollout — 2026-10-19

A primeira execução encontrou o backend A saudável, mas o gate HTTP do host expirou antes do roteamento. O cleanup foi confirmado. A e B passam a ficar sem portas publicadas na rede interna; os gates fazem HTTP dentro dos containers e preservam identidade, score e HTTP 422. Só o NGINX publica loopback por uma bridge adicional de entrada. A validação Docker da correção ainda está pendente.


## Blue-green: aceite nativo e decisão — 2026-10-19

**O gate descrito como pendente nas células de preparação está concluído.** O autor aprovou a troca A → B → A pela mesma URL NGINX, com bloqueio do pacote inválido antes da troca, falha de B observada como HTTP 502 e retorno verificado em **3,10 s**. B e o retorno reproduziram o score **0.4178630794049542**. Pesos, imagem e bytes de entrada preservados; recursos temporários removidos.

Validação informada: **110 testes de serving aprovados e 2 opt-in ignorados** (pytest 4,28 s; tox 6,94 s); teste Docker **1 aprovado em 21,42 s** (tox 23,90 s). Os hashes do controller/checker e os pins foram conciliados com o código/evidência; o assistente não acessou o engine ou o recibo nativo independentemente.

A correção separou os backends privados da entrada do gateway e foi aprovada nessa execução. Mantemos **blue-green** para o portfolio: planejar uma réplica ativa, candidata durante a entrega e anterior disponível por uma janela de rollback a definir. Essa política permanente ainda não existe: hoje o ensaio remove todos os seus recursos. Canary depende de tráfego e observação representativos.

Próximo incremento: conectar imagem/release fixadas, gates, promoção explícita e rollback ao fluxo versionado de entrega; depois provisionamento KinD + Helm. Este gate não valida novo modelo/imagem, CD de produção, zero downtime ou SLA; 3,10 s é uma observação local.

[Procedimento e limites](../../docs/operations/SERVING_ROLLOUT.md#aceite-nativo--2026-10-19) · [Evidência do autor](../../references/evidence/serving_rollout_author_validation_2026-10-19.json). Não é necessário repetir o ensaio para aplicar este fechamento documental.
